# State Space Duality (SSD) — from scan to structured matmul

This notebook derives the Mamba-2 **SSD** form step-by-step and verifies
the algebra numerically against a naive sequential scan.

## 1. Mamba-2 recurrence (per head)

Mamba-2 simplifies Mamba-1's per-(d_inner, d_state) matrix `A` to a
**scalar per head** `A_h`. `B` and `C` are shared across heads within a
group. Let `H` = nheads, `P` = headdim, `N` = dstate, `L` = seqlen.

Inputs (one batch, one head, group $g(h)$):

$$ x_t \in \mathbb{R}^P,\; \delta_t \in \mathbb{R},\; B_t, C_t \in \mathbb{R}^N,\; A \in \mathbb{R}_- $$

Discretized step (ZOH-style):

$$
\begin{aligned}
\bar A_t &= \exp(\delta_t A) \quad\text{(scalar)} \\
h_t &= \bar A_t\, h_{t-1} + \delta_t B_t\, x_t^{\top} \quad (\text{shape } P \times N) \\
y_t &= h_t C_t \quad (\text{shape } P)
\end{aligned}
$$

State $h_t$ is $P \times N$ per head; since `A` is scalar the recurrence
on $h_t$ is **rank-1-update-friendly**.

## 2. Unroll the recurrence

Define cumulative log-decay $a_t = \sum_{u \le t} \delta_u A$ and
$\alpha_{i\to j} = \exp(a_j - a_i) = \prod_{u=i+1}^{j} \bar A_u$.

Unrolling the scan (assume $h_0 = 0$):

$$
h_j = \sum_{i \le j} \alpha_{i \to j}\, \delta_i\, B_i\, x_i^{\top}
$$

$$
y_j = C_j^{\top} h_j = \sum_{i \le j}\underbrace{\alpha_{i \to j}}_{\text{decay}}\, \delta_i\, \underbrace{(C_j^{\top} B_i)}_{\text{scalar}}\, x_i
$$

This is a **dot-product of a causal weighted sum** — structurally identical
to masked attention with a specific low-rank decay pattern. That's the
**duality**.

## 3. Matrix form — one chunk

Stack the sequence within a chunk of length $S$. Let $X \in \mathbb{R}^{S \times P}$,
$B, C \in \mathbb{R}^{S \times N}$, $\delta \in \mathbb{R}^S$. Define the
**structured lower-triangular decay matrix**

$$
L_{ji} = \begin{cases} \exp(a_j - a_i) & i \le j \\ 0 & i > j \end{cases}
$$

and the **token-interaction matrix** $M = L \odot (C B^{\top})$ (shape $S \times S$).
Then

$$
Y_{\text{intra}} = M \cdot \mathrm{diag}(\delta) \cdot X
$$

That's a **structured matmul**: $O(S^2 P)$ FLOPs — GEMM-bound, not
memory-bound. On modern hardware this hits tensor cores.

## 4. Across chunks — state passing

Split the sequence into $C$ chunks of size $S$. Within a chunk we just
showed the output is a matmul **given $h$ at the chunk's left edge**.

Per-chunk **final state** is another matmul:

$$
s_c = \sum_{i \in \text{chunk } c} \alpha_{i \to \text{end}(c)}\, \delta_i\, B_i\, x_i^{\top} \in \mathbb{R}^{P \times N}
$$

States propagate across chunks with a **second** structured matmul over
a $C \times C$ lower-triangular decay matrix of *chunk-total* log-decays
$\alpha_{\text{chunk}, c}= \exp(\sum_{t \in c} \delta_t A)$.

Finally the carry-in from chunk $c-1$ adds to intra-chunk output:

$$
y_j^{(c)} = y_{j,\text{intra}}^{(c)} + \exp(a_j - a_{\text{start}(c)})\, C_j^{\top} s_{c-1}
$$

So the **whole scan** decomposes into three einsums:

1. `chunk_state` — (B, x, δ, decay) → per-chunk final states
2. `state_passing` — structured matmul across chunks
3. `chunk_scan` — intra-chunk output + carry-in

This is exactly what `src/mamba_minimal/scan_ssd.py` implements.

## 5. Numerical verification

Build a toy sequence, run a **naive scalar scan** and the **SSD chunked**
form, and verify they agree to fp32 roundoff.

In [1]:
import torch
torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

B, L, H, P, G, N = 1, 128, 4, 16, 2, 16
chunk_size = 32

x  = torch.randn(B, L, H, P, device=device)
dt = torch.rand(B, L, H, device=device) * 0.5
A  = -torch.rand(H, device=device).abs() - 0.1
Bt = torch.randn(B, L, G, N, device=device)
Ct = torch.randn(B, L, G, N, device=device)


def naive_scan(x, dt, A, Bt, Ct):
    '''Sequential Mamba-2 recurrence, bit-exact reference.'''
    B_, L_, H_, P_ = x.shape
    N_ = Bt.shape[-1]
    G_ = Bt.shape[-2]
    h = torch.zeros(B_, H_, P_, N_, device=x.device)
    y = torch.zeros_like(x)
    for t in range(L_):
        dA = torch.exp(dt[:, t] * A)               # (B, H)
        g = torch.arange(H_, device=x.device) // (H_ // G_)
        Bi = Bt[:, t, g, :]                        # (B, H, N)
        Ci = Ct[:, t, g, :]                        # (B, H, N)
        h = dA[..., None, None] * h + (dt[:, t][..., None, None] * Bi[:, :, None, :] * x[:, t][..., None])
        y[:, t] = (h * Ci[:, :, None, :]).sum(dim=-1)
    return y


y_naive = naive_scan(x, dt, A, Bt, Ct)

import sys, pathlib
sys.path.insert(0, str(pathlib.Path('.').resolve().parent / 'src'))
from mamba_minimal.scan_ssd import selective_scan_ssd

y_ssd = selective_scan_ssd(x=x, dt=dt, A=A, B=Bt, C=Ct, chunk_size=chunk_size)

diff = (y_naive - y_ssd).abs().max().item()
print('max |naive - SSD| =', diff)
assert diff < 1e-4, 'SSD must match naive scan'
print('SSD form ≡ sequential scan (to fp32 roundoff). ✅')

max |naive - SSD| = 5.4836273193359375e-06
SSD form ≡ sequential scan (to fp32 roundoff). ✅


## 6. Why it's fast

| Form | FLOPs | Hardware hit |
|---|---|---|
| Sequential scan | $O(L \cdot P \cdot N)$ | memory-bound, no parallelism along `L` |
| Parallel scan (Blelloch) | $O(L \cdot P \cdot N)$ | memory-bound, parallel but with depth-$\log L$ launches |
| **SSD (this form)** | $O(L \cdot S \cdot P)$ intra + $O(C \cdot C \cdot H \cdot P \cdot N)$ inter | **GEMM-bound** — tensor cores |

The trick: pick $S \approx \sqrt{L}$ so the two matmul costs balance.
For $L = 2048$, $S = 64$ is typical. That's why Mamba-2 via SSD
approaches attention-class throughput on modern GPUs — we've turned a
scan into a GEMM.